# REST + MCP + LLM 集成教程 (v0.2.1)

本教程帮助你理解如何将 REST API、MCP Server 和 LLM 串联起来，构建完整的 AI Agent 应用。

## 学习目标

1. 理解三层架构：REST API → MCP Server → LLM Agent
2. 掌握 FastAPI 创建 REST API
3. 学会用 FastMCP 封装 REST API 为 MCP 工具
4. 理解 LLM 如何通过自然语言调用整个链路

In [1]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "mcp", "openai", "python-dotenv", "fastapi", "uvicorn", "httpx", "-q"
])
print("依赖安装完成！")

依赖安装完成！


In [2]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

项目根目录: d:\SVN\RogueRabbit
源码路径: d:\SVN\RogueRabbit\src


## 1. 架构概览

```
┌──────────────┐     HTTP      ┌──────────────┐     STDIO     ┌──────────────┐     API      ┌──────────────┐
│    用户      │ ────────────► │   LLM Agent  │ ────────────► │  MCP Server  │ ───────────► │   REST API   │
│  (自然语言)  │               │    (GLM)     │               │  (FastMCP)   │              │  (FastAPI)   │
└──────────────┘               └──────────────┘               └──────────────┘              └──────────────┘
```

### 各层职责

| 层 | 技术 | 职责 |
|---|------|------|
| **REST API** | FastAPI | 提供标准 HTTP 接口 |
| **MCP Server** | FastMCP | 将 HTTP 接口封装为 AI 可调用的工具 |
| **LLM Agent** | GLM + ReAct | 理解自然语言，决定调用哪些工具 |

### 数据流

1. 用户用自然语言提问："帮我创建一个新物品"
2. LLM 理解意图，决定调用 `create_item` 工具
3. MCP Server 执行工具，内部调用 REST API
4. REST API 返回结果，层层传递回用户

## 2. 第一层：REST API (FastAPI)

REST API 是最底层，提供标准的 HTTP 接口。

In [3]:
# 查看项目中的 REST API 实现
rest_app_path = project_root / "src" / "rogue_rabbit" / "apps" / "rest" / "app.py"
print(f"REST API 文件: {rest_app_path}")
print("\n" + "="*60)
with open(rest_app_path, "r", encoding="utf-8") as f:
    print(f.read())

REST API 文件: d:\SVN\RogueRabbit\src\rogue_rabbit\apps\rest\app.py

"""
FastAPI REST 应用 - 物品管理 API

提供简单的 CRUD 操作，用于演示 LLM 通过 MCP 调用 REST API。
"""

from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

# ========================================
# 数据模型
# ========================================


class ItemBase(BaseModel):
    """物品基础模型"""

    name: str
    price: float
    quantity: int = 0


class Item(ItemBase):
    """物品完整模型（包含 ID）"""

    id: int


class ItemCreate(ItemBase):
    """创建物品的请求模型"""

    pass


# ========================================
# 内存数据库
# ========================================

_items_db: dict[int, dict] = {}
_item_id_counter = 0


def _init_sample_data():
    """初始化示例数据"""
    global _item_id_counter
    _items_db[1] = {"id": 1, "name": "Apple", "price": 1.5, "quantity": 100}
    _items_db[2] = {"id": 2, "name": "Banana", "price": 2.0, "quantity": 50}
    _items_db[3] = {"id": 3, "name": "Orang

### REST API 关键点

1. **Pydantic 模型**：定义请求/响应数据结构
2. **路由装饰器**：`@app.get`, `@app.post`, `@app.put`, `@app.delete`
3. **生命周期管理**：`lifespan` 函数管理启动/关闭逻辑

In [4]:
# 创建一个简化的 REST API 用于演示
from fastapi import FastAPI
from pydantic import BaseModel

# 数据模型
class Item(BaseModel):
    id: int
    name: str
    price: float
    quantity: int = 0

# 创建应用
app = FastAPI(title="Demo REST API", version="0.1.0")

# 内存数据库
_db: dict[int, dict] = {
    1: {"id": 1, "name": "Apple", "price": 1.5, "quantity": 100},
    2: {"id": 2, "name": "Banana", "price": 2.0, "quantity": 50},
}
_id_counter = 3

@app.get("/items/")
async def list_items():
    """获取所有物品"""
    return list(_db.values())

@app.get("/items/{item_id}")
async def get_item(item_id: int):
    """获取单个物品"""
    if item_id not in _db:
        return {"error": "Item not found"}
    return _db[item_id]

@app.post("/items/")
async def create_item(item: dict):
    """创建物品"""
    global _id_counter
    item["id"] = _id_counter
    _db[_id_counter] = item
    _id_counter += 1
    return item

print("REST API 定义完成！")
print(f"路由: {[route.path for route in app.routes]}")

REST API 定义完成！
路由: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/items/', '/items/{item_id}', '/items/']


## 3. 第二层：MCP Server (FastMCP)

MCP Server 将 REST API 封装为 AI 可调用的工具。

In [5]:
# 查看项目中的 MCP Server 实现
mcp_server_path = project_root / "src" / "rogue_rabbit" / "servers" / "rest_mcp_server.py"
print(f"MCP Server 文件: {mcp_server_path}")
print("\n" + "="*60)
with open(mcp_server_path, "r", encoding="utf-8") as f:
    print(f.read())

MCP Server 文件: d:\SVN\RogueRabbit\src\rogue_rabbit\servers\rest_mcp_server.py

"""
REST API MCP Server - 将 REST API 封装为 MCP 工具

这个 MCP Server 连接到 RogueRabbit REST API，提供以下工具：
- list_items: 获取所有物品列表
- get_item: 获取单个物品详情
- create_item: 创建新物品
- update_item: 更新物品信息
- delete_item: 删除物品

运行方式:
    python -m rogue_rabbit.servers.rest_mcp_server

或作为 STDIO MCP 服务器被其他程序调用。
"""

import os

# 配置 REST API 地址
REST_API_BASE_URL = os.environ.get("REST_API_URL", "http://127.0.0.1:8000")

# 使用 FastMCP 创建 MCP Server
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("rogue-rabbit-rest")


# ========================================
# 工具定义
# ========================================


@mcp.tool()
async def list_items() -> str:
    """
    获取所有物品列表。

    返回所有可用物品的 ID、名称、价格和库存信息。
    """
    import httpx

    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(f"{REST_API_BASE_URL}/items/", timeout=5.0)
            if response.status_code == 200:
                

### MCP Server 关键点

1. **FastMCP**：简化 MCP Server 创建的框架
2. **@mcp.tool() 装饰器**：定义工具，自动生成 schema
3. **httpx.AsyncClient**：异步 HTTP 客户端调用 REST API

In [6]:
# 演示 FastMCP 的工具定义
from mcp.server.fastmcp import FastMCP

# 创建 MCP Server 实例
mcp = FastMCP("demo-server")

# 定义工具 - 使用装饰器
@mcp.tool()
def calculate(expression: str) -> str:
    """
    计算数学表达式。
    
    参数:
        expression: 数学表达式，如 '2+3*4'
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"结果: {result}"
    except Exception as e:
        return f"计算错误: {e}"

@mcp.tool()
def greet(name: str, formal: bool = False) -> str:
    """
    生成问候语。
    
    参数:
        name: 被问候者的名字
        formal: 是否使用正式语气（默认 False）
    """
    if formal:
        return f"尊敬的 {name}，您好！"
    return f"你好，{name}！"

print("MCP Server 工具定义完成！")
print("工具会自动生成 JSON Schema 供 LLM 理解")

MCP Server 工具定义完成！
工具会自动生成 JSON Schema 供 LLM 理解


## 4. 第三层：LLM Agent (ReAct)

LLM Agent 是最上层，理解自然语言并决定调用哪些工具。

In [7]:
# 查看项目中的 ReAct Agent 实现
agent_path = project_root / "src" / "rogue_rabbit" / "core" / "react_agent.py"
print(f"ReAct Agent 文件: {agent_path}")
print("\n" + "="*60)
with open(agent_path, "r", encoding="utf-8") as f:
    content = f.read()
    # 只显示前 80 行
    lines = content.split("\n")[:80]
    print("\n".join(lines))
    print("\n... (更多内容请查看源文件)")

ReAct Agent 文件: d:\SVN\RogueRabbit\src\rogue_rabbit\core\react_agent.py

"""
ReAct Agent - 基于 Reasoning + Acting 模式的智能代理

ReAct = Reasoning + Acting
- LLM 分析问题，决定使用什么工具
- 执行工具调用
- 根据结果继续推理
- 直到得出最终答案
"""

import json
import re

from rogue_rabbit.contracts import Message, Role
from rogue_rabbit.contracts.mcp import MCPTool


class ReActAgent:
    """
    ReAct Agent - 使用 LLM 进行推理和工具调用

    使用方式:
        agent = ReActAgent(llm_client, mcp_client)
        answer = await agent.run("你的问题")
    """

    def __init__(
        self,
        llm_client,
        mcp_client,
        max_iterations: int = 10,
        verbose: bool = True,
    ):
        """
        初始化 ReAct Agent

        参数:
            llm_client: LLM 客户端（需实现 complete 方法）
            mcp_client: MCP 客户端（需实现 list_tools, call_tool 方法）
            max_iterations: 最大迭代次数
            verbose: 是否打印详细日志
        """
        self._llm = llm_client
        self._mcp = mcp_client
        self._max_iterations = max_iterations
        self.

### ReAct 模式

ReAct = **Re**asoning + **Act**ing

```
┌─────────────────────────────────────────────────────────┐
│                    ReAct 循环                           │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  1. THOUGHT: 分析问题，决定下一步                       │
│  2. ACTION: 选择工具并准备参数                          │
│  3. OBSERVATION: 执行工具，获取结果                     │
│  4. 重复直到可以给出 ANSWER                             │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

In [8]:
# ReAct Agent 工作流程演示
print("""
ReAct Agent 工作流程示例
========================

用户问题: "请查看有哪些物品，然后创建一个新物品叫 Grape"

--- 轮次 1 ---
THOUGHT: 首先需要查看现有物品列表
ACTION: list_items
ARGUMENTS: {}

OBSERVATION: 
- ID: 1, 名称: Apple, 价格: 1.5, 库存: 100
- ID: 2, 名称: Banana, 价格: 2.0, 库存: 50

--- 轮次 2 ---
THOUGHT: 已了解现有物品，现在创建 Grape
ACTION: create_item
ARGUMENTS: {"name": "Grape", "price": 3.0, "quantity": 30}

OBSERVATION: 成功创建物品: ID=3, 名称=Grape

--- 轮次 3 ---
THOUGHT: 任务完成
ANSWER: 已查看物品列表并成功创建了 Grape 物品
""")


ReAct Agent 工作流程示例

用户问题: "请查看有哪些物品，然后创建一个新物品叫 Grape"

--- 轮次 1 ---
THOUGHT: 首先需要查看现有物品列表
ACTION: list_items
ARGUMENTS: {}

OBSERVATION: 
- ID: 1, 名称: Apple, 价格: 1.5, 库存: 100
- ID: 2, 名称: Banana, 价格: 2.0, 库存: 50

--- 轮次 2 ---
THOUGHT: 已了解现有物品，现在创建 Grape
ACTION: create_item
ARGUMENTS: {"name": "Grape", "price": 3.0, "quantity": 30}

OBSERVATION: 成功创建物品: ID=3, 名称=Grape

--- 轮次 3 ---
THOUGHT: 任务完成
ANSWER: 已查看物品列表并成功创建了 Grape 物品



## 5. 完整流程演示

让我们模拟完整的调用流程。

In [9]:
import asyncio
import httpx

async def demo_http_call():
    """演示 HTTP 调用 REST API"""
    print("演示：使用 httpx 调用 REST API")
    print("="*50)
    
    # 注意：这需要 REST 服务在运行
    # 这里只是演示代码结构
    
    demo_code = '''
async with httpx.AsyncClient() as client:
    # 1. 获取物品列表
    response = await client.get("http://127.0.0.1:8000/items/")
    items = response.json()
    print(f"物品列表: {items}")
    
    # 2. 创建新物品
    response = await client.post(
        "http://127.0.0.1:8000/items/",
        json={"name": "Grape", "price": 3.0, "quantity": 30}
    )
    new_item = response.json()
    print(f"创建的物品: {new_item}")
    
    # 3. 获取单个物品
    response = await client.get(f"http://127.0.0.1:8000/items/{new_item['id']}")
    item = response.json()
    print(f"物品详情: {item}")
'''
    print(demo_code)
    print("\n这就是 MCP Server 内部做的事情！")

await demo_http_call()

演示：使用 httpx 调用 REST API

async with httpx.AsyncClient() as client:
    # 1. 获取物品列表
    response = await client.get("http://127.0.0.1:8000/items/")
    items = response.json()
    print(f"物品列表: {items}")

    # 2. 创建新物品
    response = await client.post(
        "http://127.0.0.1:8000/items/",
        json={"name": "Grape", "price": 3.0, "quantity": 30}
    )
    new_item = response.json()
    print(f"创建的物品: {new_item}")

    # 3. 获取单个物品
    response = await client.get(f"http://127.0.0.1:8000/items/{new_item['id']}")
    item = response.json()
    print(f"物品详情: {item}")


这就是 MCP Server 内部做的事情！


## 6. 启动完整演示

要运行完整的三层架构演示，需要：

1. 启动 REST 服务（后台运行）
2. 启动 MCP Server（通过 STDIO）
3. 运行 LLM Agent

In [10]:
# 运行完整实验的命令
print("""
运行完整演示
============

在终端中运行:

    python -m rogue_rabbit.experiments.07_rest_mcp_llm

这个实验会：
1. 在后台启动 REST API 服务 (端口 8000)
2. 连接到 MCP Server (STDIO 模式)
3. 使用 GLM 进行自然语言交互

预期输出:
- REST 服务启动成功
- MCP Server 连接成功，发现 5 个工具
- LLM 调用工具完成任务
- 返回最终答案
""")


运行完整演示

在终端中运行:

    python -m rogue_rabbit.experiments.07_rest_mcp_llm

这个实验会：
1. 在后台启动 REST API 服务 (端口 8000)
2. 连接到 MCP Server (STDIO 模式)
3. 使用 GLM 进行自然语言交互

预期输出:
- REST 服务启动成功
- MCP Server 连接成功，发现 5 个工具
- LLM 调用工具完成任务
- 返回最终答案



In [11]:
# 检查环境配置
import os
from dotenv import load_dotenv

env_path = project_root / ".env"
load_dotenv(env_path)

api_key = os.environ.get("ZHIPU_API_KEY")
if api_key:
    print("✅ ZHIPU_API_KEY 已配置")
else:
    print("❌ ZHIPU_API_KEY 未配置")
    print(f"\n请在 {env_path} 中添加:")
    print("ZHIPU_API_KEY=your-api-key")

✅ ZHIPU_API_KEY 已配置


## 7. 关键代码模式总结

In [12]:
print("""
关键代码模式
============

1. REST API (FastAPI)
--------------------
@app.get("/items/")
async def list_items():
    return items

@app.post("/items/")
async def create_item(item: ItemModel):
    # 创建逻辑
    return new_item


2. MCP Server (FastMCP)
----------------------
mcp = FastMCP("server-name")

@mcp.tool()
async def list_items() -> str:
    async with httpx.AsyncClient() as client:
        response = await client.get("http://api/items/")
        return format_result(response.json())


3. LLM Agent (ReAct)
-------------------
agent = ReActAgent(llm_client, mcp_client)
answer = await agent.run("用户问题")

# Agent 内部循环:
# 1. LLM 分析问题，决定工具调用
# 2. 执行工具，获取结果
# 3. 将结果反馈给 LLM
# 4. 重复直到得出最终答案
""")


关键代码模式

1. REST API (FastAPI)
--------------------
@app.get("/items/")
async def list_items():
    return items

@app.post("/items/")
async def create_item(item: ItemModel):
    # 创建逻辑
    return new_item


2. MCP Server (FastMCP)
----------------------
mcp = FastMCP("server-name")

@mcp.tool()
async def list_items() -> str:
    async with httpx.AsyncClient() as client:
        response = await client.get("http://api/items/")
        return format_result(response.json())


3. LLM Agent (ReAct)
-------------------
agent = ReActAgent(llm_client, mcp_client)
answer = await agent.run("用户问题")

# Agent 内部循环:
# 1. LLM 分析问题，决定工具调用
# 2. 执行工具，获取结果
# 3. 将结果反馈给 LLM
# 4. 重复直到得出最终答案



## 总结

### 三层架构

| 层 | 技术 | 关键概念 |
|---|------|----------|
| REST API | FastAPI | 路由、Pydantic 模型、生命周期 |
| MCP Server | FastMCP | 工具定义、httpx 调用、STDIO 通信 |
| LLM Agent | GLM + ReAct | 思考-行动循环、工具解析 |

### 核心理解

1. **分层解耦**：每层只关心自己的职责
2. **标准化协议**：MCP 让工具集成变得统一
3. **自然语言接口**：用户不需要知道底层实现

### 扩展方向

- 添加认证和授权
- 支持数据库持久化
- 添加更多 REST API 端点
- 优化 Agent 的提示词

### 下一步

- 运行 `experiments/07_rest_mcp_llm.py` 体验完整流程
- 阅读 `apps/rest/app.py` 理解 REST API 实现
- 阅读 `servers/rest_mcp_server.py` 理解 MCP Server 实现
- 阅读 `core/react_agent.py` 理解 Agent 逻辑